<p align="center">
  <img src="../media/adifind_logo_cropped.png" width="420" />
</p>

# AdiFind Quickstart

**AdiFind** is a deep-learning pipeline for **automated adipocyte (fat cell) detection** in
gigapixel whole-slide histology images (`.svs`, `.ndpi`, `.tiff`+++++). AdiFind uses
[Detectron2](https://github.com/facebookresearch/detectron2) instance segmentation to find
every adipocyte, measure its area, and — optionally — compute its distance to the nearest
tumour boundary.

### What this notebook covers

| Section | Description |
|---|---|
| **1. Verify Installation** | Check that PyTorch, Detectron2 and OpenSlide are available |
| **2. Download Models** | Pull pre-trained weights from HuggingFace (one-time) |
| **3. Process a Slide** | Run the full detection pipeline on an example `.svs` file |
| **4. View Results** | Display the annotated thumbnail overlay |
| **5. CSV Results** | Explore per-adipocyte measurements and size distributions |
| **6. QuPath Integration** | Export GeoJSON annotations for QuPath |
| **7. Batch Processing** | Process a folder of slides and compare results |

### Prerequisites

Install AdiFind following the instructions in `documentation/INSTALL.md`, or run:
```bash
conda env create -f environment.yml
conda activate adifind
```

> **Hardware:** GPU inference uses a compatible PyTorch/Detectron2 GPU backend.
> The packaged examples in this repo use CUDA/NVIDIA, and some ROCm-capable AMD setups may also work where the upstream stack supports them.
> CPU-only mode is supported, but it is usually slower.

## 1. Verify Installation

AdiFind depends on three core libraries:

- **PyTorch** — deep learning framework (with CUDA for GPU acceleration)
- **Detectron2** — Facebook AI's instance segmentation library
- **OpenSlide** — reads vendor-specific whole-slide image formats (`.svs`, `.ndpi`, etc.)

Run the cell below to confirm they are importable and check GPU availability.

In [ ]:
import sys, os
from pathlib import Path

# AdiFind expects to run from code/ — locate it if we're not already there
if not Path('main.py').exists():
    for _p in [Path('code'), Path('..', 'code')]:
        if (_p / 'main.py').exists():
            os.chdir(_p)
            break

# --- PyTorch ---
import torch
print(f'PyTorch {torch.__version__}  |  GPU backend available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Accelerator: {torch.cuda.get_device_name(0)}')

# --- Detectron2 ---
import detectron2
print(f'Detectron2 {detectron2.__version__}')

# --- OpenSlide ---
# On Windows the OpenSlide DLLs must be explicitly registered.
# On Linux/macOS they are found via the system library path automatically.
OPENSLIDE_PATH = os.environ.get("OPENSLIDE_PATH", r"C:\OpenSlide\bin")
if hasattr(os, "add_dll_directory") and os.path.isdir(OPENSLIDE_PATH):
    os.add_dll_directory(OPENSLIDE_PATH)

import openslide
print(f'OpenSlide {openslide.__library_version__}')

print('\n\u2705 All core dependencies are available.')

## 2. Download or Use Local Models

AdiFind uses three tailored-made Detectron2 Mask R-CNN models. The notebook can
download them from [HuggingFace](https://huggingface.co/letarg/adifind) and cache
them locally (`~/.cache/adifind/models/` on Linux,
`%LOCALAPPDATA%\adifind\models\` on Windows) when the repo is accessible.

The current HuggingFace model repo is private, so the recommended no-token path is
to set `local_model_dir` in the next cell to a folder containing these standard
filenames:

| Model | Purpose | Filename | Size |
|---|---|---|---|
| **Adipocyte** | Detect and segment individual fat cells (required) | `adifind_adipocyte.pth` | ~856 MB |
| **Tissue Guidance** | Detect tissue regions to skip empty glass on slide (optional) | `adifind_tissue_guidance.pth` | ~856 MB |
| **Tumor** | Segment tumour regions for distance-to-tumour analysis (optional) | `adifind_tumor.pth` | ~856 MB |

In [ ]:
from pathlib import Path
from model_downloader import ensure_model, get_cache_dir

cache_dir = Path(get_cache_dir())

model_filenames = {
    'adipocyte': 'adifind_adipocyte.pth',
    'tissue': 'adifind_tissue_guidance.pth',
    'tumor': 'adifind_tumor.pth',
}

# Recommended for the current private-repo setup: set this to a folder containing
# all three canonical checkpoint files. Set to None only if you have Hugging Face
# access and want to use the cached/downloaded path.
# Example: local_model_dir = Path('/path/to/adifind_models')
local_model_dir = None

if local_model_dir is not None:
    local_model_dir = Path(local_model_dir).expanduser().resolve()

    if not local_model_dir.is_dir():
        raise FileNotFoundError(f'Local model directory does not exist: {local_model_dir}')

    model_paths = {
        name: local_model_dir / filename
        for name, filename in model_filenames.items()
    }
    missing_files = [path.name for path in model_paths.values() if not path.exists()]
    if missing_files:
        missing_list = ', '.join(missing_files)
        raise FileNotFoundError(
            f'Local model directory is missing required checkpoint file(s): {missing_list}'
        )

    model_source = 'local'
else:
    model_paths = {
        name: Path(ensure_model(name, checkpoint=filename))
        for name, filename in model_filenames.items()
    }
    model_source = 'cached/downloaded'

print(f'Model source: {model_source}')
print(f"Adipocyte model: {model_paths['adipocyte']}")
print(f"Tissue model:    {model_paths['tissue']}")
print(f"Tumor model:     {model_paths['tumor']}")
if model_source == 'local':
    print(f'\nLocal model directory: {local_model_dir}')
else:
    print(f'\nModel cache directory: {cache_dir}')

print('\nModel status:')
for name, path in model_paths.items():
    status = 'available' if path.exists() else 'missing'
    print(f' - {name}: {status} ({path.name})')

## 3. Process an Example Slide

AdiFind processes a whole-slide image through a **sliding-window** approach:

1. **Tissue guidance** (optional) — a lightweight model scans a low-resolution thumbnail
   to find tissue regions, so the main model can skip empty glass.
2. **Adipocyte detection** — the main Mask R-CNN model runs on each tissue-containing
   window at full resolution.
3. **Merging** — overlapping detections across adjacent windows are merged using a
   union-find algorithm with IoU thresholds.
4. **Tumour distance** (optional) — a second model segments tumour regions, then a
   Euclidean distance transform measures each adipocyte’s proximity to the nearest tumour.

### Configuration

The cell below sets up all processing parameters via an `argparse.Namespace` — the
same flags you would pass on the command line. `update_config_from_args(args)` then
translates them into the global `config` singleton, so **every toggle is set in one
place only**.

The most important tunables are:

| Parameter | Effect |
|---|---|
| `tissue_guidance` | Skip empty-glass windows, effective for slides where samples are deposted with empty space between them |
| `tumor_segmentation` | Compute distance from each adipocyte to nearest tumour |
| `roi_freehand` | Launch an interactive ROI selector before processing |
| `CHUNK_WORKERS` | Number of parallel threads for chunk processing (set on `config`) |
| `MAX_IO_WORKERS` | I/O thread-pool size for reading slide tiles (set on `config`) |
| `BATCH_INFERENCE_SIZE` | How many windows to batch per GPU forward pass (set on `config`) |

GPU controls are split so you can disable them independently:

- `disable_gpu_accel=True` forces CPU-only processing.
- `disable_gpu_ops=True` keeps GPU inference enabled, but disables CuPy ops, GPU preprocessing, and GPU label mapping.
- `disable_gpu_preprocessing=True` disables only GPU Sobel/inversion preprocessing.

> **Window size & stride** are chosen automatically based on the slide’s microns-per-pixel
> (MPP). The sentinel values `[2048, 2048]` / `[1024, 1024]` signal “use automatic sizing”.

In [ ]:
from pathlib import Path
import argparse
from main import process_single_image
from config import config, paths
from configuration_manager import update_config_from_args

# ── Performance tuning ───────────────────────────────────────────────
# Adjust these based on your hardware. Higher values use more RAM/VRAM.
# These have no CLI equivalent, so they must be set on config directly.
config.CHUNK_WORKERS = 12        # parallel workers for chunk processing
config.MAX_IO_WORKERS = 25       # I/O thread-pool size
config.BATCH_INFERENCE_SIZE = 4  # windows per GPU inference batch
config

# ── Processing arguments ─────────────────────────────────────────────
# These mirror the CLI flags of `python main.py`.
# update_config_from_args() translates them into config.* settings,
# so every toggle only needs to be set once — right here.
#
# Window size & stride use sentinel values → AdiFind picks optimal sizes
# from the slide’s MPP:  MPP ≥ 0.40 → 1100/900,  MPP < 0.40 → 2000/1700
args = argparse.Namespace(
    
    # ── Core ──────────────────────────────────────────────────────
    window_size=[2048, 2048],      # sentinel → automatic MPP-based sizing
    stride=[1024, 1024],           # sentinel → automatic MPP-based sizing
    output_dir='../quickstart_output',  # relative to code/ → saves at repo root
    gpu_id=0,                      # which GPU to use (0 = first)
    
    # ── Feature toggles ──────────────────────────────────────────
    tissue_guidance=True,          # skip empty-glass windows on slides with large blank regions
    tumor_segmentation=True,       # run tumour segmentation model
    save_distance_map=False,        # save a distance-coloured overlay image
    save_tissue_window_grid=True, # save a debug image of the tissue grid
    extended_properties=False,     # extra morphological measurements (slower)
    
    # ── Display ──────────────────────────────────────────────────
    annotated_scale=0.3,           # thumbnail scale factor for the annotated overlay
    save_mode='fast',              # one of 'fast', 'balanced', or 'high_quality'
    show_adipocyte_ids=False,      # draw numeric IDs on each adipocyte in the overlay
    show_grid=False,               # draw sliding-window grid lines on the overlay
    
    # ── ROI guidance ─────────────────────────────────────────────
    roi_freehand=False,            # set True to launch interactive ROI selector
    roi_polygon_file=None,         # path to a pre-saved ROI polygon JSON file
    roi_max_dim=2048,              # max thumbnail dimension for ROI selection GUI
    roi_min_coverage=0.2,          # minimum tissue coverage to keep a window (0-1)
    
    # ── Advanced ─────────────────────────────────────────────────
    debug=None,                    # 'processed' or 'unprocessed' for debug outputs
    disable_gpu_accel=False,       # True = CPU-only mode (disable GPU inference and non-inference GPU paths)
    disable_gpu_ops=False,         # True = keep GPU inference, disable CuPy ops, GPU preprocessing, and GPU label mapping
    disable_gpu_preprocessing=False, # True = disable only GPU Sobel/inversion preprocessing
    low_memory=False,              # enable for machines with ≤ 64 GB RAM
    memmap_mask=False,             # disk-backed mask (very large slides)
    
    # ── Internal defaults (required by update_config_from_args) ──
    config_file=None, min_area=None, max_area=None, batch_size=None,
    profiling=False, hide_adipocyte_ids=False, hide_grid=False,
)

# Bridge args → config singleton (tissue_guidance, tumor_segmentation, etc.)
update_config_from_args(args)

# Remember the original model settings so notebook reruns can restore them.
default_model_dirs = globals().get('default_model_dirs')
if default_model_dirs is None:
    default_model_dirs = {
        'adipocyte': paths.ADIPOCYTE_MODEL_DIR,
        'tissue': paths.TISSUE_MODEL_DIR,
        'tumor': paths.TUMOR_MODEL_DIR,
    }
    globals()['default_model_dirs'] = default_model_dirs

# Reuse the local model folder from section 2 when provided.
local_model_dir = globals().get('local_model_dir')
if local_model_dir is not None:
    local_model_dir = Path(local_model_dir).expanduser().resolve()
    paths.ADIPOCYTE_MODEL_DIR = str(local_model_dir)
    paths.TISSUE_MODEL_DIR = str(local_model_dir)
    paths.TUMOR_MODEL_DIR = str(local_model_dir)
    print(f'Using local model directory: {local_model_dir}')
else:
    paths.ADIPOCYTE_MODEL_DIR = default_model_dirs['adipocyte']
    paths.TISSUE_MODEL_DIR = default_model_dirs['tissue']
    paths.TUMOR_MODEL_DIR = default_model_dirs['tumor']

# ── Run the pipeline ─────────────────────────────────────────────────
slide_path = '../example_data/K106942.svs'
results = process_single_image(slide_path, args, '../quickstart_output')

print(f"\n{'='*50}")
print(f"Detected {results['total_adipocytes']} adipocytes")
print(f"Processing time: {results['total_time']:.1f}s")

## 4. View the Annotated Thumbnail

AdiFind produces an annotated thumbnail of the entire slide with every detected adipocyte
outlined. This gives a quick visual sanity check of the detection quality.

In [ ]:
import glob
from PIL import Image
import matplotlib.pyplot as plt

# WSI annotated thumbnails can be very large — lift PIL's default pixel limit
Image.MAX_IMAGE_PIXELS = None

# Locate the annotated thumbnail in the output directory
output_dir = results['output_dir']
thumbnails = glob.glob(os.path.join(output_dir, '*_adifind_annotated*'))

if thumbnails:
    img = Image.open(thumbnails[0])

    # Downscale for notebook display if wider/taller than 4096 px
    max_display = 4096
    if max(img.size) > max_display:
        scale = max_display / max(img.size)
        img = img.resize((int(img.width * scale), int(img.height * scale)), Image.LANCZOS)

    fig, ax = plt.subplots(1, 1, figsize=(14, 10))
    ax.imshow(img)
    ax.set_title(f'Annotated Thumbnail \u2014 {results["total_adipocytes"]} adipocytes detected')
    ax.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print('No annotated thumbnail found in output directory.')

## 5. Explore the CSV Results

Every detected adipocyte is written to a CSV with per-cell measurements including:

| Column | Description |
|---|---|
| `Area_Microns_Squared` | Adipocyte cross-sectional area in µm² |
| `Centroid_X` / `Centroid_Y` | Pixel coordinates of the cell centre |
| `Distance_To_Closest_Tumour` | Distance (µm) to the nearest tumour boundary (if tumour model was enabled) |
| `Distance_Bin` | Categorical bin: Close (≤ 100 µm), Medium (≤ 500 µm), Far (> 500 µm) |

Below we load the CSV, show summary statistics, and plot size distributions.
If tumour segmentation was enabled, a **viridis-coloured stacked histogram** shows how
adipocyte size varies with distance from the tumour.

In [ ]:
import pandas as pd
import numpy as np

# Find the per-adipocyte CSV in the output folder
csv_files = glob.glob(os.path.join(output_dir, 'adipocyte_information_*'))

if csv_files:
    df = pd.read_csv(csv_files[0])
    AREA_COL = 'Area_Microns_Squared'
    DIST_COL = 'Distance_To_Closest_Tumour'

    print(f'Results table: {len(df)} adipocytes, {len(df.columns)} measurements\n')

    # Quick summary statistics
    if AREA_COL in df.columns:
        print(f"Median adipocyte size: {df[AREA_COL].median():.1f} \u00b5m\u00b2")
        print(f"Mean adipocyte size:   {df[AREA_COL].mean():.1f} \u00b5m\u00b2\n")

    display(df.head(10))

    # ── Adipocyte size histogram ──────────────────────────────────────
    fig, ax = plt.subplots(1, 1, figsize=(10, 5))
    if AREA_COL in df.columns:
        df[AREA_COL].hist(bins=50, ax=ax, edgecolor='white', color='steelblue')
        ax.set_xlabel('Adipocyte Area (\u00b5m\u00b2)')
    ax.set_ylabel('Count')
    ax.set_title('Adipocyte Size Distribution')
    plt.tight_layout()
    plt.show()

    # ── Tumour distance vs adipocyte size (viridis stacked bar) ───────
    # Each colour band represents a distance bin from the tumour boundary.
    # Darker colours = closer to tumour; lighter = further away.
    if DIST_COL in df.columns and AREA_COL in df.columns:
        from matplotlib.colors import Normalize
        from matplotlib.cm import ScalarMappable

        df_valid = df.dropna(subset=[DIST_COL, AREA_COL])
        if len(df_valid) > 0:
            max_dist = df_valid[DIST_COL].quantile(0.99)
            n_bins = 8
            bin_edges = np.linspace(0, max_dist, n_bins + 1)
            df_valid = df_valid[df_valid[DIST_COL] <= max_dist].copy()
            df_valid['dist_bin'] = pd.cut(df_valid[DIST_COL], bins=bin_edges)

            size_bins = np.linspace(df_valid[AREA_COL].quantile(0.01),
                                    df_valid[AREA_COL].quantile(0.99), 40)
            cmap = plt.cm.viridis
            norm = Normalize(vmin=0, vmax=n_bins - 1)

            fig, ax = plt.subplots(1, 1, figsize=(12, 6))
            bottom = np.zeros(len(size_bins) - 1)

            for i, interval in enumerate(sorted(df_valid['dist_bin'].dropna().unique())):
                subset = df_valid[df_valid['dist_bin'] == interval][AREA_COL]
                counts, _ = np.histogram(subset, bins=size_bins)
                color = cmap(norm(i))
                label = f'{interval.left:.0f}\u2013{interval.right:.0f} \u00b5m'
                ax.bar((size_bins[:-1] + size_bins[1:]) / 2, counts,
                       width=np.diff(size_bins) * 0.9, bottom=bottom,
                       color=color, edgecolor='none', label=label)
                bottom += counts

            ax.set_xlabel('Adipocyte Area (\u00b5m\u00b2)')
            ax.set_ylabel('Count')
            ax.set_title('Adipocyte Size Distribution by Distance from Tumour')
            sm = ScalarMappable(cmap=cmap, norm=Normalize(
                vmin=bin_edges[0], vmax=bin_edges[-1]))
            sm.set_array([])
            cbar = plt.colorbar(sm, ax=ax, pad=0.02)
            cbar.set_label('Distance from Tumour (\u00b5m)')
            plt.tight_layout()
            plt.show()
    else:
        print('\nNo tumour distance data \u2014 run with tumor_segmentation=True to see distance analysis.')
else:
    print('No CSV results found in output directory.')

## 6. QuPath Integration

AdiFind exports [GeoJSON](https://geojson.org/) annotations that can be imported directly
into [QuPath](https://qupath.github.io/), the popular open-source tool for digital
pathology image analysis.

**To import into QuPath:**

1. Open the same WSI in QuPath
2. **File → Import Objects → GeoJSON**
3. Select the `*_qupath_annotations.geojson` file from the output directory

Each detected adipocyte appears as a polygon annotation with its area measurement attached.

In [ ]:
import json

# Check for the GeoJSON export in the output directory
geojson_files = glob.glob(os.path.join(output_dir, '*_qupath_annotations.geojson'))

if geojson_files:
    with open(geojson_files[0]) as f:
        geojson = json.load(f)
    n_features = len(geojson.get('features', []))
    print(f'GeoJSON file contains {n_features} annotation features.')
    print(f'File: {geojson_files[0]}')
else:
    print('No GeoJSON file found. Ensure ENABLE_QUPATH_EXPORT is True in config.')

## 7. Batch Processing — Process a Folder of Slides

For cohort-level studies you typically need to process many slides at once.
AdiFind can iterate over an entire folder of WSIs, creating a **separate output
directory** for each image. Results can then, as exemplified in the below cell,
then be combined into a single DataFrame
for cross-slide comparison.

The cell below processes every supported image in `example_data/` using the
same settings from Section 3.

In [ ]:
from argument_parser import get_image_files
from pathlib import Path

# Discover all supported slide files (.svs, .ndpi, .tiff, …)
slide_folder = '../example_data/'
slide_files = get_image_files(slide_folder)
print(f'Found {len(slide_files)} slides in {slide_folder}:\n')
for f in slide_files:
    print(f'  \u2022 {os.path.basename(f)}')

# Each image gets its own subdirectory under the batch output root
batch_output_root = '../quickstart_batch_output'
batch_results = []

for i, slide in enumerate(slide_files, 1):
    name = Path(slide).stem                    # e.g. "K106942"
    print(f'\n{"="*60}')
    print(f'[{i}/{len(slide_files)}] Processing {name}...')
    print(f'{"="*60}')

    # Create a per-image output folder: ../quickstart_batch_output/K106942/
    image_output_dir = os.path.join(batch_output_root, name)
    os.makedirs(image_output_dir, exist_ok=True)

    result = process_single_image(slide, args, image_output_dir)
    batch_results.append(result)
    print(f'  \u2705 {result["total_adipocytes"]} adipocytes in {result["total_time"]:.1f}s')

# ── Summary table ─────────────────────────────────────────────────────
print(f'\n\n{"="*60}')
print('BATCH SUMMARY')
print(f'{"="*60}')
summary_df = pd.DataFrame([{
    'Image': r['image_name'],
    'Adipocytes': r['total_adipocytes'],
    'Windows': r['total_windows'],
    'Tumors': r.get('num_tumors', 0),
    'Time (s)': round(r['total_time'], 1),
} for r in batch_results])
display(summary_df)
total_adipo = summary_df['Adipocytes'].sum()
total_time = summary_df['Time (s)'].sum()
print(f'\nTotal: {total_adipo} adipocytes across {len(slide_files)} slides in {total_time:.1f}s')

### Combined Batch Results

Below we load every per-image CSV into a single DataFrame for cross-slide analysis.
The visualisations include:

- **Per-image annotated thumbnails** — one per slide
- **Side-by-side histograms** — adipocyte size distributions per slide (easy comparison)
- **Overlay histogram** — all slides superimposed for direct shape comparison
- **Tumour distance plots** — per-slide viridis stacked bars (if tumour data is available)

In [ ]:
AREA_COL = 'Area_Microns_Squared'
DIST_COL = 'Distance_To_Closest_Tumour'

# ── Load and concatenate all per-image CSVs ──────────────────────────
all_dfs = []
for r in batch_results:
    csv_files = glob.glob(os.path.join(r['output_dir'], 'adipocyte_information_*'))
    if csv_files:
        img_df = pd.read_csv(csv_files[0])
        img_df['Image'] = r['image_name']  # tag each row with its source slide
        all_dfs.append(img_df)

if not all_dfs:
    print('No CSV results found.')
else:
    combined = pd.concat(all_dfs, ignore_index=True)
    print(f'Combined: {len(combined)} adipocytes across {len(all_dfs)} slides\n')

    # ── Per-slide summary statistics ──────────────────────────────────
    if AREA_COL in combined.columns:
        stats = combined.groupby('Image')[AREA_COL].agg(['count', 'median', 'mean', 'std'])
        stats.columns = ['Count', 'Median (\u00b5m\u00b2)', 'Mean (\u00b5m\u00b2)', 'Std (\u00b5m\u00b2)']
        display(stats.round(1))

    # ── Per-image annotated thumbnails ────────────────────────────────
    for r in batch_results:
        thumbs = glob.glob(os.path.join(r['output_dir'], '*_adifind_annotated*'))
        if thumbs:
            img = Image.open(thumbs[0])
            max_display = 4096
            if max(img.size) > max_display:
                s = max_display / max(img.size)
                img = img.resize((int(img.width * s), int(img.height * s)), Image.LANCZOS)
            fig, ax = plt.subplots(1, 1, figsize=(14, 10))
            ax.imshow(img)
            ax.set_title(f'{r["image_name"]} \u2014 {r["total_adipocytes"]} adipocytes')
            ax.axis('off')
            plt.tight_layout()
            plt.show()

    # ── Per-image size distributions (side by side) ───────────────────
    if AREA_COL in combined.columns:
        images = combined['Image'].unique()
        n = len(images)
        fig, axes = plt.subplots(1, n, figsize=(7 * n, 5), squeeze=False)
        for idx, name in enumerate(images):
            ax = axes[0, idx]
            subset = combined[combined['Image'] == name]
            subset[AREA_COL].hist(bins=50, ax=ax, edgecolor='white', color='steelblue')
            ax.set_xlabel('Adipocyte Area (\u00b5m\u00b2)')
            ax.set_ylabel('Count')
            ax.set_title(name)
        plt.suptitle('Adipocyte Size Distribution \u2014 Per Slide', y=1.02, fontsize=14)
        plt.tight_layout()
        plt.show()

    # ── Combined overlay histogram ────────────────────────────────────
    # All slides on the same axes for direct shape comparison
    if AREA_COL in combined.columns:
        fig, ax = plt.subplots(1, 1, figsize=(10, 5))
        for name, group in combined.groupby('Image'):
            group[AREA_COL].hist(bins=50, ax=ax, alpha=0.5, edgecolor='white', label=name)
        ax.set_xlabel('Adipocyte Area (\u00b5m\u00b2)')
        ax.set_ylabel('Count')
        ax.set_title('Adipocyte Size Distribution \u2014 All Slides (overlay)')
        ax.legend()
        plt.tight_layout()
        plt.show()

    # ── Per-image tumour distance plots (viridis stacked bars) ────────
    if DIST_COL in combined.columns and AREA_COL in combined.columns:
        from matplotlib.colors import Normalize
        from matplotlib.cm import ScalarMappable

        for img_name in combined['Image'].unique():
            df_img = combined[combined['Image'] == img_name].dropna(subset=[DIST_COL, AREA_COL])
            if len(df_img) == 0:
                print(f'\n{img_name}: no tumours detected \u2014 skipping distance plot')
                continue

            max_dist = df_img[DIST_COL].quantile(0.99)
            n_bins = 8
            bin_edges = np.linspace(0, max_dist, n_bins + 1)
            df_img = df_img[df_img[DIST_COL] <= max_dist].copy()
            df_img['dist_bin'] = pd.cut(df_img[DIST_COL], bins=bin_edges)

            size_bins = np.linspace(df_img[AREA_COL].quantile(0.01),
                                    df_img[AREA_COL].quantile(0.99), 40)
            cmap = plt.cm.viridis
            norm = Normalize(vmin=0, vmax=n_bins - 1)

            fig, ax = plt.subplots(1, 1, figsize=(12, 6))
            bottom = np.zeros(len(size_bins) - 1)

            for i, interval in enumerate(sorted(df_img['dist_bin'].dropna().unique())):
                subset = df_img[df_img['dist_bin'] == interval][AREA_COL]
                counts, _ = np.histogram(subset, bins=size_bins)
                color = cmap(norm(i))
                label = f'{interval.left:.0f}\u2013{interval.right:.0f} \u00b5m'
                ax.bar((size_bins[:-1] + size_bins[1:]) / 2, counts,
                       width=np.diff(size_bins) * 0.9, bottom=bottom,
                       color=color, edgecolor='none', label=label)
                bottom += counts

            ax.set_xlabel('Adipocyte Area (\u00b5m\u00b2)')
            ax.set_ylabel('Count')
            ax.set_title(f'Adipocyte Size by Distance from Tumour \u2014 {img_name}')
            sm = ScalarMappable(cmap=cmap, norm=Normalize(
                vmin=bin_edges[0], vmax=bin_edges[-1]))
            sm.set_array([])
            cbar = plt.colorbar(sm, ax=ax, pad=0.02)
            cbar.set_label('Distance from Tumour (\u00b5m)')
            plt.tight_layout()
            plt.show()
    elif config.ENABLE_TUMOR_SEGMENTATION:
        print('\nTumour segmentation was enabled but no tumours were detected in these slides.')
    else:
        print('\nTumour segmentation disabled \u2014 enable with tumor_segmentation=True for distance analysis.')